In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import os
import lightgbm as lgb
import joblib
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix

### Load training dataset:

In [6]:
#Check location of training dataset
path = "../Pre_Process/Training_df_2026-07-08.csv"
print(os.path.exists(path))
print(os.getcwd())

True
e:\AS9Wa\MSc Thesis\AgenticVision\Model\Training


In [4]:
#Import dataset
df = pd.read_csv(path)
df.head()

,session_id,label_llm,label_workload,win_idx,flow_elapsed_s,flow_position_norm,pkt_count,fwd_count,bwd_count,fwd_bwd_pkt_ratio,...,tok_rate,tok_iat_cv,delta_tok_iat_mean,delta_tok_iat_std,delta_tok_iat_p50,delta_tok_iat_ac_lag1,delta_tok_rate,delta_tok_iat_cv,bytes_per_token,iat_tok_pkt_ratio
0,58cb5f85,groq_qwen3_32b,short_factual,0,0.0,0.0000,3.0,1.0,2.0,0.5000,...,376.0,5.9401,0.000000,0.000000,0.000000,0.0000,0.0,0.0000,5.69,0.0055
1,58cb5f85,groq_qwen3_32b,short_factual,1,0.1,0.0249,2.0,1.0,1.0,1.0000,...,426.0,6.3935,-0.000340,-0.000965,-0.000083,0.0032,50.0,0.4534,3.89,0.0000
2,58cb5f85,groq_qwen3_32b,short_factual,2,0.2,0.0499,2.0,1.0,1.0,1.0000,...,434.0,6.4570,-0.000045,-0.000143,-0.000003,-0.0006,8.0,0.0635,3.82,0.0000
3,58cb5f85,groq_qwen3_32b,short_factual,3,0.3,0.0748,2.0,1.0,1.0,1.0000,...,484.0,1.4376,-0.000954,-0.012822,0.000002,-0.0445,50.0,-5.0194,3.43,0.0000
4,58cb5f85,groq_qwen3_32b,short_factual,4,0.4,0.0998,10.0,1.0,9.0,0.1111,...,484.0,1.4376,0.000000,0.000000,0.000000,0.0000,0.0,0.0000,9.57,0.0258


### Train LightGBM across all features:

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(path)

TARGET = "label_workload"
DROP_COLS = ["session_id", "label_llm", "label_model", "label_workload", "tls_detected"]

feature_cols = [c for c in df.columns if c not in DROP_COLS]
X = df[feature_cols].copy()
groups = df["session_id"].values

le = LabelEncoder()
y = le.fit_transform(df[TARGET])
class_names = le.classes_
print("Classes:", list(class_names))
print("Feature count:", X.shape[1])
print()

# ---------------------------------------------------------------------------
# 2. Grouped, stratified cross-validation
# ---------------------------------------------------------------------------
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_f1s = []
all_true, all_pred = [], []
importances = np.zeros(X.shape[1])

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # sanity check: no session leaks across the split
    train_sessions = set(groups[train_idx])
    test_sessions = set(groups[test_idx])
    assert train_sessions.isdisjoint(test_sessions), "Session leakage across folds!"

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(class_names),
        class_weight="balanced",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    print(X_train.columns.tolist())
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )

    preds = model.predict(X_test)
    fold_f1 = f1_score(y_test, preds, average="macro")
    fold_f1s.append(fold_f1)
    print(f"Fold {fold}: macro-F1 = {fold_f1:.4f}  "
          f"(train sessions={len(train_sessions)}, test sessions={len(test_sessions)})")

    all_true.extend(y_test)
    all_pred.extend(preds)
    importances += model.feature_importances_

print()
print(f"Mean macro-F1 across {N_SPLITS} folds: {np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f})")
print()

# ---------------------------------------------------------------------------
# 3. Aggregate report across all folds (out-of-fold predictions)
# ---------------------------------------------------------------------------
print("Classification report (out-of-fold, pooled across all 5 folds):")
print(classification_report(all_true, all_pred, target_names=class_names))

print("Confusion matrix (rows=true, cols=predicted):")
cm = confusion_matrix(all_true, all_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df)
print()

# ---------------------------------------------------------------------------
# 4. Feature importance (averaged across folds)
# ---------------------------------------------------------------------------
importances /= N_SPLITS
imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
print("Top 20 most important features:")
print(imp_df.head(20).to_string(index=False))

imp_df.to_csv("label_llm_feature_importance.csv", index=False)

today = datetime.today().strftime("%Y-%m-%d")
mean_fold_f1_score = (np.mean(fold_f1s).round(4))

joblib.dump(model, f"../Model_Store/workload_classifier_model_f1_{mean_fold_f1_score}_date_{today}.joblib")

Classes: ['chain_of_thought', 'classification', 'code_synthesis', 'instruction_following', 'long_generation', 'roleplay_agent', 'short_factual', 'structured_output', 'summarisation', 'translation']
Feature count: 109

['win_idx', 'flow_elapsed_s', 'flow_position_norm', 'pkt_count', 'fwd_count', 'bwd_count', 'fwd_bwd_pkt_ratio', 'total_bytes', 'fwd_bytes', 'bwd_bytes', 'byte_ratio_bwd_fwd', 'pkt_ratio_bwd_fwd', 'pktlen_mean', 'pktlen_std', 'pktlen_min', 'pktlen_max', 'pktlen_skew', 'pktlen_kurt', 'pktlen_p10', 'pktlen_p25', 'pktlen_p50', 'pktlen_p75', 'pktlen_p90', 'pktlen_p95', 'pktlen_p95_p50_ratio', 'pktlen_p75_p25_ratio', 'pktlen_iqr', 'pktlen_entropy', 'pktlen_norm_entropy', 'fwd_pktlen_mean', 'fwd_pktlen_min', 'fwd_pktlen_max', 'bwd_pktlen_mean', 'bwd_pktlen_std', 'bwd_pktlen_min', 'bwd_pktlen_max', 'bwd_pktlen_skew', 'bwd_pktlen_kurt', 'throughput_bps', 'bwd_throughput_bps', 'iat_mean', 'iat_std', 'iat_min', 'iat_max', 'iat_skew', 'iat_kurt', 'iat_p10', 'iat_p25', 'iat_p50', 'iat

['../Model_Store/llm_classifier_model_f1_0.1373_date_2026-07-08.joblib']

In [7]:
"""
Approach B: session-level aggregation for label_workload.
 
Instead of classifying individual windows, aggregate every session's windows
into ONE feature vector per session (mean/std/min/max/last of every raw
feature), producing 172 rows total -- one per session. Then a plain
StratifiedKFold is valid (no grouping needed, since each row already IS
a session, there's no risk of the same session appearing in both train/test).
"""
 
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import lightgbm as lgb
 
RANDOM_STATE = 42
N_SPLITS = 5
 
df = pd.read_csv(path)
 
TARGET = "label_workload"
NON_FEATURE_COLS = ["session_id", "label_llm", "label_model", "label_workload", "tls_detected"]
raw_feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
 
# ---------------------------------------------------------------------------
# 1. Aggregate each session's windows into one row.
#    mean/std/min/max capture the overall distribution of each stat across
#    the session; "last" captures the final/steady-state window, which can
#    differ meaningfully from the average (e.g. tapering token rate at the end).
# ---------------------------------------------------------------------------
agg_funcs = ["mean", "std", "min", "max", "last"]
session_agg = df.groupby("session_id")[raw_feature_cols].agg(agg_funcs)
session_agg.columns = ["_".join(col) for col in session_agg.columns]
session_agg = session_agg.fillna(0)  # std of a single-window session -> NaN -> 0
 
# also add session length itself as a feature (number of windows)
session_len = df.groupby("session_id").size().rename("n_windows")
session_agg = session_agg.join(session_len)
 
# attach the label (one per session, already confirmed unique earlier)
labels = df.groupby("session_id")[TARGET].first()
session_agg = session_agg.join(labels)
 
print(f"Session-level table: {session_agg.shape[0]} rows (sessions), {session_agg.shape[1]-1} features")
print()
 
# ---------------------------------------------------------------------------
# 2. Train/eval with plain StratifiedKFold (no grouping needed anymore)
# ---------------------------------------------------------------------------
X = session_agg.drop(columns=[TARGET]).copy()
X.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in X.columns]
feature_cols = list(X.columns)
 
le = LabelEncoder()
y = le.fit_transform(session_agg[TARGET])
class_names = le.classes_
 
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
 
fold_f1s = []
all_true, all_pred = [], []
importances = np.zeros(X.shape[1])
 
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
 
    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(class_names),
        class_weight="balanced",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=15,       # smaller trees -- only 137-138 training rows per fold
        min_child_samples=5, # allow smaller leaf splits given tiny dataset
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )
 
    preds = model.predict(X_test)
    fold_f1 = f1_score(y_test, preds, average="macro")
    fold_f1s.append(fold_f1)
    print(f"Fold {fold}: macro-F1 = {fold_f1:.4f}  (train={len(train_idx)}, test={len(test_idx)})")
 
    all_true.extend(y_test)
    all_pred.extend(preds)
    importances += model.feature_importances_
 
print()
print(f"Mean macro-F1 across {N_SPLITS} folds: {np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f})")
print()
 
print("Classification report (out-of-fold, pooled across all 5 folds):")
print(classification_report(all_true, all_pred, target_names=class_names, zero_division=0))
 
print("Confusion matrix (rows=true, cols=predicted):")
cm = confusion_matrix(all_true, all_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df.to_string())
print()
 
importances /= N_SPLITS
imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
print("Top 20 most important features:")
print(imp_df.head(20).to_string(index=False))
 
today = datetime.today().strftime("%Y-%m-%d")
mean_fold_f1_score = (np.mean(fold_f1s).round(4))
joblib.dump(model, f"../Model_Store/workload_classifier_model_f1_{mean_fold_f1_score}_date_{today}.joblib")


Session-level table: 172 rows (sessions), 546 features

Fold 1: macro-F1 = 0.0964  (train=137, test=35)
Fold 2: macro-F1 = 0.1742  (train=137, test=35)
Fold 3: macro-F1 = 0.1084  (train=138, test=34)
Fold 4: macro-F1 = 0.1828  (train=138, test=34)
Fold 5: macro-F1 = 0.2257  (train=138, test=34)

Mean macro-F1 across 5 folds: 0.1575 (+/- 0.0484)

Classification report (out-of-fold, pooled across all 5 folds):
                       precision    recall  f1-score   support

     chain_of_thought       0.00      0.00      0.00        16
       classification       0.00      0.00      0.00         8
       code_synthesis       0.34      0.36      0.35        28
instruction_following       0.00      0.00      0.00        12
      long_generation       0.46      0.54      0.50        24
       roleplay_agent       0.00      0.00      0.00         8
        short_factual       0.54      0.58      0.56        36
    structured_output       0.14      0.12      0.13        16
        summarisatio

['../Model_Store/workload_classifier_model_f1_0.1575_date_2026-07-08.joblib']

### Train LightGBM model, removing window reatures

In [39]:
"""
Permutation importance for label_llm, computed within each grouped CV fold.

Ends by writing out a list of "safe to drop" feature names (original CSV
column names, not sanitized) so they can be pasted straight into DROP_COLS
for future training scripts.
"""

import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
import lightgbm as lgb

RANDOM_STATE = 42
N_SPLITS = 5

df = pd.read_csv(path)

TARGET = "label_llm"
BASE_DROP_COLS = ["session_id", "label_llm", "label_model", "label_workload", "tls_detected",
                  "win_idx", "flow_position_norm", "flow_elapsed_s"] # these as flow position not needed

feature_cols = [c for c in df.columns if c not in BASE_DROP_COLS]
X = df[feature_cols].copy()

# sanitize names for LightGBM, keep a map back to the original column names
sanitized_to_original = {re.sub(r"[^A-Za-z0-9_]+", "_", c): c for c in feature_cols}
X.columns = list(sanitized_to_original.keys())

groups = df["session_id"].values
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
class_names = le.classes_

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

perm_means = np.zeros(X.shape[1])
perm_stds = np.zeros(X.shape[1])

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = lgb.LGBMClassifier(
        objective="multiclass", num_class=len(class_names), class_weight="balanced",
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbosity=-1,
    )
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])

    result = permutation_importance(
        model, X_test, y_test,
        scoring="f1_macro", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
    )
    perm_means += result.importances_mean
    perm_stds += result.importances_std
    print(f"Fold {fold} done.")

perm_means /= N_SPLITS
perm_stds /= N_SPLITS

imp_df = pd.DataFrame({
    "feature_sanitized": list(X.columns),
    "feature": [sanitized_to_original[c] for c in X.columns],
    "perm_importance_mean": perm_means,
    "perm_importance_std": perm_stds,
})
imp_df = imp_df.sort_values("perm_importance_mean", ascending=False)
imp_df.to_csv("label_llm_permutation_importance.csv", index=False)

print()
print("Top 15 features:")
print(imp_df.head(15)[["feature", "perm_importance_mean", "perm_importance_std"]].to_string(index=False))

# ---------------------------------------------------------------------------
# Build the "safe to drop" list.
# A feature is kept only if its importance is both positive AND clearly above
# its own noise band (mean > std) -- otherwise it's indistinguishable from a
# shuffled/random feature and is a candidate to drop.
# ---------------------------------------------------------------------------
imp_df["significant"] = (imp_df["perm_importance_mean"] > 0) & \
                         (imp_df["perm_importance_mean"] > imp_df["perm_importance_std"])

features_to_drop = imp_df.loc[~imp_df["significant"], "feature"].tolist()
features_to_keep = imp_df.loc[imp_df["significant"], "feature"].tolist()

print()
print(f"Keeping {len(features_to_keep)} significant features, dropping {len(features_to_drop)}.")

# Save the drop list as its own CSV
pd.Series(features_to_drop, name="feature").to_csv("label_llm_features_to_drop.csv", index=False)

print()
print("Saved: label_llm_permutation_importance.csv")
print("Saved: label_llm_features_to_drop.csv")

Fold 1 done.
Fold 2 done.
Fold 3 done.
Fold 4 done.
Fold 5 done.

Top 15 features:
              feature  perm_importance_mean  perm_importance_std
    delta_tok_iat_std              0.138969             0.006838
       delta_tok_rate              0.123690             0.005568
     delta_tok_iat_cv              0.061941             0.004254
          token_count              0.044624             0.003312
          tok_iat_p95              0.044422             0.002539
delta_tok_iat_ac_lag1              0.006160             0.001877
       byte_growth_r2              0.005619             0.001023
    delta_tok_iat_p50              0.005180             0.002528
            pkt_count              0.004420             0.000795
 pktlen_p75_p25_ratio              0.003415             0.000838
            bwd_bytes              0.003218             0.000682
          iat_entropy              0.002476             0.001264
              iat_p90              0.001996             0.000571
       

### Train model using only important features:

In [44]:
RANDOM_STATE = 42
N_SPLITS = 5

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(path)

TARGET = "label_llm"
#Import list of cols to remove:
drp_cols = pd.read_csv("label_llm_features_to_drop.csv")
drp_cols = drp_cols["feature"].tolist()

DROP_COLS = ["session_id","win_idx","flow_position_norm", "flow_elapsed_s","label_llm", "label_model", "label_workload", "tls_detected"] + drp_cols

feature_cols = [c for c in df.columns if c not in DROP_COLS]
X = df[feature_cols].copy()
groups = df["session_id"].values

le = LabelEncoder()
y = le.fit_transform(df[TARGET])
class_names = le.classes_
print("Classes:", list(class_names))
print("Feature count:", X.shape[1])
print()

# ---------------------------------------------------------------------------
# 2. Grouped, stratified cross-validation
# ---------------------------------------------------------------------------
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_f1s = []
all_true, all_pred = [], []
importances = np.zeros(X.shape[1])

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # sanity check: no session leaks across the split
    train_sessions = set(groups[train_idx])
    test_sessions = set(groups[test_idx])
    assert train_sessions.isdisjoint(test_sessions), "Session leakage across folds!"

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(class_names),
        class_weight="balanced",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    print(X_train.columns.tolist())
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )

    preds = model.predict(X_test)
    fold_f1 = f1_score(y_test, preds, average="macro")
    fold_f1s.append(fold_f1)
    print(f"Fold {fold}: macro-F1 = {fold_f1:.4f}  "
          f"(train sessions={len(train_sessions)}, test sessions={len(test_sessions)})")

    all_true.extend(y_test)
    all_pred.extend(preds)
    importances += model.feature_importances_

print()
print(f"Mean macro-F1 across {N_SPLITS} folds: {np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f})")
print()

# ---------------------------------------------------------------------------
# 3. Aggregate report across all folds (out-of-fold predictions)
# ---------------------------------------------------------------------------
print("Classification report (out-of-fold, pooled across all 5 folds):")
print(classification_report(all_true, all_pred, target_names=class_names))

print("Confusion matrix (rows=true, cols=predicted):")
cm = confusion_matrix(all_true, all_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df)
print()

# ---------------------------------------------------------------------------
# 4. Feature importance (averaged across folds)
# ---------------------------------------------------------------------------
importances /= N_SPLITS
imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
print("Top 20 most important features:")
print(imp_df.head(20).to_string(index=False))

imp_df.to_csv("label_llm_feature_importance.csv", index=False)

from datetime import datetime
today = datetime.today().strftime("%Y-%m-%d")
mean_fold_f1_score = (np.mean(fold_f1s).round(4))

joblib.dump(model, f"../Model_Store/llm_classifier_model_imp_features_only_f1_{mean_fold_f1_score}_date_{today}.joblib")

Classes: ['gemini_3.1_flash-lite', 'groq_llama-3.3-70b-versatile', 'groq_qwen3_32b', 'openai/gpt-oss-120b']
Feature count: 27

['pkt_count', 'bwd_count', 'bwd_bytes', 'pktlen_mean', 'pktlen_p10', 'pktlen_p25', 'pktlen_p75', 'pktlen_p75_p25_ratio', 'bwd_throughput_bps', 'iat_p90', 'iat_entropy', 'iat_ac_lag2', 'byte_growth_r2', 'byte_growth_residual_std', 'token_count', 'tok_iat_kurt', 'tok_iat_p95', 'tok_iat_iqr', 'tok_iat_entropy', 'tok_iat_ac_lag1', 'tok_rate', 'delta_tok_iat_std', 'delta_tok_iat_p50', 'delta_tok_iat_ac_lag1', 'delta_tok_rate', 'delta_tok_iat_cv', 'bytes_per_token']
Fold 1: macro-F1 = 0.9064  (train sessions=140, test sessions=32)
['pkt_count', 'bwd_count', 'bwd_bytes', 'pktlen_mean', 'pktlen_p10', 'pktlen_p25', 'pktlen_p75', 'pktlen_p75_p25_ratio', 'bwd_throughput_bps', 'iat_p90', 'iat_entropy', 'iat_ac_lag2', 'byte_growth_r2', 'byte_growth_residual_std', 'token_count', 'tok_iat_kurt', 'tok_iat_p95', 'tok_iat_iqr', 'tok_iat_entropy', 'tok_iat_ac_lag1', 'tok_rate', '

['../Model_Store/llm_classifier_model_imp_features_only_f1_0.911_date_2026-07-08.joblib']